# Iris_Bayesian_decision_making

Bản chuyển từ sample **0. Getting Started/Iris_Bayesian_decision_making.ipynb**. Chạy các ô theo thứ tự.

Sample dùng GaussianNB của scikit-learn. Bản tensor tính cùng quy tắc Bayes Gauss độc lập và so sánh từng dự đoán với GaussianNB trên cùng phép chia 60/40.

In [1]:
from sklearn.datasets import load_iris
from sklearn.naive_bayes import GaussianNB
import torch

iris = load_iris()
x = torch.tensor(iris.data, dtype=torch.float64)
y = torch.tensor(iris.target)
order = torch.randperm(len(y), generator=torch.Generator().manual_seed(2018))
train, test = order[:90], order[90:]
classes = torch.unique(y[train])
scores = []
epsilon = 1e-9 * x[train].var(0, unbiased=False).max()
for k in classes:
    samples = x[train][y[train] == k]
    mean = samples.mean(0)
    print("Lớp", k.item(), "| trung bình", mean.tolist(), "| hiệp phương sai", samples.T.cov())
    var = samples.var(0, unbiased=False) + epsilon
    prior = len(samples) / len(train)
    log_density = -0.5 * (torch.log(2 * torch.pi * var) + (x[test] - mean).square() / var)
    scores.append(torch.tensor(prior).log() + log_density.sum(1))
pred = classes[torch.stack(scores).argmax(0)]
reference = GaussianNB().fit(x[train].numpy(), y[train].numpy()).predict(x[test].numpy())
assert torch.equal(pred, torch.from_numpy(reference))
print(f"Đúng {(pred == y[test]).sum().item()}/60 | accuracy {(pred == y[test]).double().mean():.2%}")

Lớp 0 | trung bình [5.0, 3.448387096774194, 1.4129032258064516, 0.23548387096774195] | hiệp phương sai tensor([[0.1373, 0.1037, 0.0177, 0.0053],
        [0.1037, 0.1539, 0.0090, 0.0042],
        [0.0177, 0.0090, 0.0258, 0.0045],
        [0.0053, 0.0042, 0.0045, 0.0110]], dtype=torch.float64)
Lớp 1 | trung bình [5.889655172413794, 2.713793103448276, 4.248275862068966, 1.3] | hiệp phương sai tensor([[0.2302, 0.0834, 0.1423, 0.0432],
        [0.0834, 0.0948, 0.0832, 0.0461],
        [0.1423, 0.0832, 0.1862, 0.0632],
        [0.0432, 0.0461, 0.0632, 0.0393]], dtype=torch.float64)
Lớp 2 | trung bình [6.64, 2.966666666666667, 5.543333333333333, 2.013333333333333] | hiệp phương sai tensor([[0.3514, 0.0717, 0.2858, 0.0305],
        [0.0717, 0.1009, 0.0477, 0.0429],
        [0.2858, 0.0477, 0.3067, 0.0456],
        [0.0305, 0.0429, 0.0456, 0.0674]], dtype=torch.float64)
Đúng 56/60 | accuracy 93.33%
